# EDA 004.05 — Correlation and Relationships

**Measuring the strength of associations between variables**

## Learning Objectives
By the end of this notebook you will be able to:
- Choose the right correlation method for different data types and distributions
- Calculate and interpret Pearson, Spearman, and Kendall correlations
- Measure association between two categorical variables with Cramér's V
- Measure association between a numerical and a binary variable (point-biserial)
- Explain why correlation ≠ causation with concrete examples

---

## Correlation Methods at a Glance

| Method | Variables | Measures | Requires | Range |
|---|---|---|---|---|
| **Pearson r** | Num × Num | Linear relationship | Normal + no outliers | −1 to +1 |
| **Spearman ρ** | Num × Num | Monotonic relationship | Ordinal/ranked | −1 to +1 |
| **Kendall τ** | Num × Num | Concordance of pairs | Small samples | −1 to +1 |
| **Cramér's V** | Cat × Cat | Association strength | — | 0 to +1 |
| **Point-biserial** | Num × Binary | Association | — | −1 to +1 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

penguins = sns.load_dataset('penguins').dropna()
titanic  = sns.load_dataset('titanic')
tips     = sns.load_dataset('tips')

print('Penguins:', penguins.shape)
penguins.head()

---
## 1 — Pearson Correlation

**Reference:** [scipy.stats.pearsonr](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html)

Pearson's $r$ measures the **linear relationship** between two continuous variables:

$$r = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum(x_i-\bar{x})^2 \sum(y_i-\bar{y})^2}}$$

| $|r|$ | Strength |
|---|---|
| 0.0 – 0.19 | Negligible |
| 0.2 – 0.39 | Weak |
| 0.4 – 0.59 | Moderate |
| 0.6 – 0.79 | Strong |
| 0.8 – 1.0 | Very strong |

> **Assumption:** Both variables should be approximately normally distributed with no extreme outliers. Use Spearman otherwise.

In [ ]:
num_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

# Pearson correlation matrix
pearson_matrix = penguins[num_cols].corr(method='pearson')

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(pearson_matrix, dtype=bool))
sns.heatmap(pearson_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', vmin=-1, vmax=1, center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Pearson Correlation Matrix — Penguins')
plt.tight_layout()
plt.show()

# With p-value
x = penguins['flipper_length_mm']
y = penguins['body_mass_g']
r, p = stats.pearsonr(x, y)
print(f'flipper_length vs body_mass: r={r:.3f}, p-value={p:.2e}')

---
## 2 — Spearman vs Pearson: Handling Non-linear Relationships

**Reference:** [scipy.stats.spearmanr](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html)

Spearman's $\rho$ is the Pearson correlation computed on the **ranks** of the data, not the raw values.

- Works for **monotonic** relationships (not just linear)
- **Robust** to outliers and non-normality
- Use when data is ordinal, or when Pearson assumptions are violated

In [ ]:
np.random.seed(42)
x = np.random.uniform(0.1, 5, 200)
y_nonlinear = np.log(x) + np.random.normal(0, 0.3, 200)   # monotonic but non-linear

r_pearson, _  = stats.pearsonr(x, y_nonlinear)
r_spearman, _ = stats.spearmanr(x, y_nonlinear)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(x, y_nonlinear, alpha=0.5, s=20)
axes[0].set_title(f'Non-linear (log) relationship\nPearson r={r_pearson:.3f} | Spearman ρ={r_spearman:.3f}')
axes[0].set_xlabel('x')
axes[0].set_ylabel('log(x) + noise')

# Spearman captures it better than Pearson
# Show that Spearman on ranks is equivalent to Pearson on ranks
rank_x = stats.rankdata(x)
rank_y = stats.rankdata(y_nonlinear)
axes[1].scatter(rank_x, rank_y, alpha=0.5, s=20, color='coral')
axes[1].set_title(f'Same data — ranked\nPearson on ranks = Spearman ρ={r_spearman:.3f}')
axes[1].set_xlabel('rank(x)')
axes[1].set_ylabel('rank(y)')

plt.tight_layout()
plt.show()

# Compare on penguins data
print("Pearson vs Spearman on Penguin measurements:")
p_mat = penguins[num_cols].corr(method='pearson')
s_mat = penguins[num_cols].corr(method='spearman')
diff  = (p_mat - s_mat).abs()
print(diff.round(3))

---
## 3 — Kendall's Tau

**Reference:** [scipy.stats.kendalltau](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kendalltau.html)

Kendall's $\tau$ measures **concordance**: for every pair of observations, is the ordering of x consistent with the ordering of y?

$$\tau = \frac{\text{concordant pairs} - \text{discordant pairs}}{\binom{n}{2}}$$

- More **interpretable** than Spearman (directly measures fraction of concordant pairs)
- Better for **small samples** and ordinal data
- Usually smaller in magnitude than Spearman for the same data

In [ ]:
x = penguins['flipper_length_mm'].values
y = penguins['body_mass_g'].values

r_p, p_p   = stats.pearsonr(x, y)
rho_s, p_s = stats.spearmanr(x, y)
tau, p_t   = stats.kendalltau(x, y)

print('flipper_length vs body_mass:')
print(f'  Pearson  r  = {r_p:.4f}  (p={p_p:.2e})')
print(f'  Spearman ρ  = {rho_s:.4f}  (p={p_s:.2e})')
print(f'  Kendall  τ  = {tau:.4f}  (p={p_t:.2e})')

print('\nInterpretation:')
n = len(x)
n_pairs = n * (n - 1) // 2
concordant = int((tau * n_pairs + n_pairs) / 2)
print(f'  Out of {n_pairs} pairs, ~{concordant} are concordant ({tau*100:.1f}% net concordance)')

---
## 4 — Cramér's V (Categorical × Categorical)

**Reference:** [scipy.stats.chi2_contingency](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html)

Pearson/Spearman/Kendall require numerical data. For two categorical variables, use **Cramér's V**:

$$V = \sqrt{\frac{\chi^2 / n}{\min(r-1, c-1)}}$$

where $\chi^2$ is the chi-squared statistic, $n$ is the sample size, and $r, c$ are the number of rows/columns in the contingency table.

| V | Association |
|---|---|
| 0.0 – 0.1 | Negligible |
| 0.1 – 0.3 | Weak |
| 0.3 – 0.5 | Moderate |
| > 0.5 | Strong |

In [ ]:
def cramers_v(x, y):
    """Compute Cramér's V association for two categorical variables."""
    ct = pd.crosstab(x, y)
    chi2, _, _, _ = stats.chi2_contingency(ct)
    n = ct.sum().sum()
    min_dim = min(ct.shape) - 1
    return np.sqrt(chi2 / (n * min_dim))

# Titanic categorical associations
cat_cols = ['pclass', 'sex', 'embarked', 'survived']
titanic_clean = titanic[cat_cols].dropna()

v_matrix = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
for c1 in cat_cols:
    for c2 in cat_cols:
        v_matrix.loc[c1, c2] = cramers_v(titanic_clean[c1], titanic_clean[c2])

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones(v_matrix.shape, dtype=bool))
sns.heatmap(v_matrix.astype(float), mask=mask, annot=True, fmt='.2f',
            cmap='YlOrRd', vmin=0, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Cramér's V — Titanic Categorical Variables")
plt.tight_layout()
plt.show()

---
## 5 — Point-Biserial (Numerical × Binary)

**Reference:** [scipy.stats.pointbiserialr](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pointbiserialr.html)

The **point-biserial correlation** measures the association between a **continuous variable** and a **binary (0/1) variable**.
Mathematically, it is equivalent to Pearson's $r$ when one variable is binary.

In [ ]:
df_clean = titanic[['age', 'fare', 'pclass', 'survived']].dropna()

print('Point-biserial correlation with survived:')
for col in ['age', 'fare', 'pclass']:
    r, p = stats.pointbiserialr(df_clean['survived'], df_clean[col])
    print(f'  {col:8s}: r={r:+.4f}  p={p:.4f}')

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ['age', 'fare', 'pclass']):
    r, _ = stats.pointbiserialr(df_clean['survived'], df_clean[col])
    sns.boxplot(data=df_clean, x='survived', y=col, ax=ax, palette='Set2')
    ax.set_title(f'{col}\n(point-biserial r={r:+.3f})')
    ax.set_xlabel('Survived')

plt.tight_layout()
plt.show()

---
## 6 — Correlation vs Causation

**Reference:** [Spurious Correlations (tylervigen.com)](https://www.tylervigen.com/spurious-correlations)

High correlation **does NOT imply causation**. The relationship might be due to:

| Scenario | Example |
|---|---|
| **Coincidence** | Ice cream sales correlate with drowning rates (both increase in summer) |
| **Confounding variable** | Shoe size correlates with reading ability in children (age is the confounder) |
| **Reverse causation** | Hospitals have more sick people → going to hospital causes sickness? |
| **Spurious correlation** | Nicolas Cage film releases correlate with pool drownings |

> To establish causation, you need a **controlled experiment (randomised trial)** or **causal inference methods** (e.g. difference-in-differences, instrumental variables).

---
## Key Takeaways

- Use **Pearson** for normally distributed, linear relationships
- Use **Spearman** for skewed data, ordinal variables, or monotonic (non-linear) relationships
- Use **Kendall** for small samples or ordinal data — more interpretable than Spearman
- Use **Cramér's V** for two categorical variables
- Use **point-biserial** for numerical vs binary — it's just Pearson in disguise
- Always check a scatter plot alongside a correlation coefficient — **Anscombe's Quartet** shows four datasets with the same Pearson $r$ but completely different patterns

---
## Further Reading

- [scipy.stats docs](https://docs.scipy.org/doc/scipy/reference/stats.html)
- [Anscombe's Quartet (Wikipedia)](https://en.wikipedia.org/wiki/Anscombe%27s_quartet) — why you must plot, not just correlate
- [Spurious Correlations](https://www.tylervigen.com/spurious-correlations) — compelling demonstration of correlation ≠ causation
- [The Book of Why](https://www.basicbooks.com/titles/judea-pearl/the-book-of-why/9780465097609/) — Judea Pearl (causal inference)
- [Practical Statistics for Data Scientists](https://www.oreilly.com/library/view/practical-statistics-for/9781492072935/) — Ch. 3